In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

n_estimators_list = [100, 200, 300, 400, 500, 800]
contamination_list = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, "auto"]


def contamination_to_str(cont):
    if isinstance(cont, str):
        return cont
    return str(cont).replace(".", "_")


data_path = Path("../data_processed/NAB/industrial_machine_features_labeled.csv")
results_root = Path("../results/iForest_NAB")
results_root.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path)

feature_cols = ["value", "diff1", "roll_mean", "roll_std", "roll_min", "roll_max"]
required_cols = feature_cols + ["y_true"]
df = df.dropna(subset=required_cols).reset_index(drop=True)

X = df[feature_cols].copy()
y_true = df["y_true"].astype(int)

X_normal = X[y_true == 0].copy()

scaler = StandardScaler()

X_train = pd.DataFrame(
    scaler.fit_transform(X_normal),
    columns=feature_cols,
    index=X_normal.index
)

X_test = pd.DataFrame(
    scaler.transform(X),
    columns=feature_cols,
    index=X.index
)

all_results = []

for contamination in contamination_list:
    contamination_dir = results_root / f"contamination_{contamination_to_str(contamination)}"
    contamination_dir.mkdir(parents=True, exist_ok=True)

    contamination_results = []

    for n_estimators in n_estimators_list:
        exp_name = f"n_estimators_{n_estimators}"
        exp_dir = contamination_dir / exp_name
        exp_dir.mkdir(parents=True, exist_ok=True)

        model = IsolationForest(
            n_estimators=n_estimators,
            max_samples="auto",
            contamination=contamination,
            max_features=1.0,
            bootstrap=False,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train)

        df_temp = df.copy()

        pred_raw = model.predict(X_test)
        decision_scores = model.decision_function(X_test)
        score_samples = model.score_samples(X_test)

        df_temp["iforest_pred_raw"] = pred_raw
        df_temp["iforest_decision"] = decision_scores
        df_temp["iforest_score"] = score_samples
        df_temp["is_anomaly"] = (df_temp["iforest_pred_raw"] == -1).astype(int)

        y_pred = df_temp["is_anomaly"]

        cm = confusion_matrix(y_true, y_pred)

        accuracy = accuracy_score(y_true, y_pred) * 100
        precision = precision_score(y_true, y_pred, zero_division=0) * 100
        recall = recall_score(y_true, y_pred, zero_division=0) * 100
        f1 = f1_score(y_true, y_pred, zero_division=0) * 100

        result_row = {
            "experiment": exp_name,
            "n_estimators": n_estimators,
            "contamination": contamination,
            "accuracy_%": round(accuracy, 2),
            "precision_%": round(precision, 2),
            "recall_%": round(recall, 2),
            "f1_%": round(f1, 2)
        }

        all_results.append(result_row)
        contamination_results.append(result_row)

        df_temp.to_csv(exp_dir / "predictions.csv", index=False)

        metrics_df = pd.DataFrame({
            "Metrika": [
                "Presnosť (Accuracy)",
                "Precíznosť (Precision)",
                "Citlivosť (Recall)",
                "F1-skóre"
            ],
            "Hodnota (%)": [
                round(accuracy, 2),
                round(precision, 2),
                round(recall, 2),
                round(f1, 2)
            ]
        })

        fig, ax = plt.subplots(figsize=(6, 2))
        ax.axis("off")

        table = ax.table(
            cellText=metrics_df.values,
            colLabels=metrics_df.columns,
            loc="center",
            cellLoc="center"
        )

        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 2)

        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_text_props(weight="bold")
                cell.set_facecolor("#dce6f1")
            else:
                cell.set_facecolor("#f9f9f9")

        plt.title("Výsledné metriky modelu Isolation Forest – NAB", fontsize=12, pad=20)
        plt.savefig(exp_dir / "metrics_table.png", dpi=300, bbox_inches="tight")
        plt.close()

        plt.figure(figsize=(6, 5))
        plt.imshow(cm, interpolation="nearest")
        plt.title("Matica zámen (Confusion Matrix)")
        plt.colorbar()

        classes = ["Normálne", "Anomálie"]
        tick_marks = np.arange(len(classes))

        plt.xticks(tick_marks, classes)
        plt.yticks(tick_marks, classes)

        threshold = cm.max() / 2.0 if cm.max() > 0 else 0.5

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                color = "black" if cm[i, j] > threshold else "white"
                plt.text(
                    j, i, cm[i, j],
                    ha="center", va="center",
                    color=color
                )

        plt.xlabel("Predikované triedy")
        plt.ylabel("Skutočné triedy")
        plt.tight_layout()
        plt.savefig(exp_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
        plt.close()

        if "timestamp" in df_temp.columns:
            x_axis = pd.to_datetime(df_temp["timestamp"])
            x_label = "Čas"
        else:
            x_axis = np.arange(len(df_temp))
            x_label = "Index"

        plt.figure(figsize=(14, 5))
        plt.plot(x_axis, df_temp["value"], linewidth=1)

        anomaly_idx = df_temp["is_anomaly"] == 1
        plt.scatter(
            np.array(x_axis)[anomaly_idx],
            df_temp.loc[anomaly_idx, "value"],
            s=18
        )

        plt.title(f"{exp_name}, contamination={contamination_to_str(contamination)}")
        plt.xlabel(x_label)
        plt.ylabel("Hodnota")
        plt.tight_layout()
        plt.savefig(exp_dir / "anomaly_plot.png", dpi=300, bbox_inches="tight")
        plt.close()

    contamination_results_df = pd.DataFrame(contamination_results)
    contamination_results_df = contamination_results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
    contamination_results_df.to_csv(contamination_dir / "summary_results.csv", index=False)

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by="f1_%", ascending=False).reset_index(drop=True)
results_df.to_csv(results_root / "summary_results.csv", index=False)

print("Done")

Done
